In [1]:

import pandas as pd

df = pd.read_csv('/content/customer_churn_raw_v2.csv')

print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)


Dataset Shape: (605, 13)

First 5 Rows:


,CustomerID,Age,Region,TenureMonths,SubscriptionType,MonthlyChargesINR,TotalChargesINR,ContractType,SupportTickets,LatePayments,PaymentMethod,AutoPay,Churn
0,CUST-100001,20,NaN,9,Pro,13097.76,unknown,Month-to-Month,1,2,Card,No,Yes
1,CUST-100002,55,Central,52,Pro,1101.18,57334.22,One Year,0,0,Card,Yes,No
2,CUST-100003,40,Central,67,Standard,692.66,44554.47,Month-to-Month,6,2,UPI,No,Yes
3,CUST-100004,53,South,48,Pro,1156.49,53956.12,One Year,2,2,Card,Yes,No
4,CUST-100005,21,West,24,Standard,754.16,18191.59,Month-to-Month,0,1,Card,Yes,No



Column Names:
['CustomerID', 'Age', 'Region', 'TenureMonths', 'SubscriptionType', 'MonthlyChargesINR', 'TotalChargesINR', 'ContractType', 'SupportTickets', 'LatePayments', 'PaymentMethod', 'AutoPay', 'Churn']

Missing Values:
CustomerID            0
Age                   0
Region               13
TenureMonths          0
SubscriptionType      0
MonthlyChargesINR     0
TotalChargesINR       0
ContractType          0
SupportTickets        0
LatePayments          0
PaymentMethod         0
AutoPay               0
Churn                 0
dtype: int64

Duplicate Rows: 5

Data Types:
CustomerID            object
Age                    int64
Region                object
TenureMonths           int64
SubscriptionType      object
MonthlyChargesINR    float64
TotalChargesINR       object
ContractType          object
SupportTickets         int64
LatePayments           int64
PaymentMethod         object
AutoPay               object
Churn                 object
dtype: object


In [2]:

import pandas as pd

# Keep original data unchanged
clean_df = df.copy()

# 1. Standardize column names
clean_df.columns = (
    clean_df.columns.str.strip()
    .str.lower()
    .str.replace(r'\s+', '_', regex=True)
)

# 2. Remove exact duplicate rows
before = len(clean_df)
clean_df = clean_df.drop_duplicates().copy()
duplicates_removed = before - len(clean_df)

# 3. Standardize text values
for col in clean_df.select_dtypes(include="object").columns:
    clean_df[col] = clean_df[col].map(
        lambda x: x.strip() if isinstance(x, str) else x
    )
    clean_df[col] = clean_df[col].replace("", pd.NA)

# 4. Convert unknown values to missing values
clean_df = clean_df.replace(
    r"(?i)^\s*unknown\s*$", pd.NA, regex=True
)

# 5. Validate numeric columns
numeric_cols = [
    "age", "tenuremonths", "monthlychargesinr",
    "totalchargesinr", "supporttickets", "latepayments"
]

for col in numeric_cols:
    clean_df[col] = pd.to_numeric(
        clean_df[col], errors="coerce"
    )

# 6. Handle missing Region values
if "region" in clean_df.columns:
    clean_df["region"] = clean_df["region"].fillna("Unknown")

# 7. Fill missing numeric values with median
for col in numeric_cols:
    if clean_df[col].isna().any():
        median_value = clean_df[col].median()
        if pd.notna(median_value):
            clean_df[col] = clean_df[col].fillna(median_value)

# 8. Document cleaning decisions
print("Cleaning Summary")
print("----------------")
print("Exact duplicate rows removed:", duplicates_removed)
print("Rows remaining:", len(clean_df))
print("\nMissing values after cleaning:")
print(clean_df.isnull().sum())
print("\nFinal data types:")
print(clean_df.dtypes)


Cleaning Summary
----------------
Exact duplicate rows removed: 5
Rows remaining: 600

Missing values after cleaning:
customerid           0
age                  0
region               0
tenuremonths         0
subscriptiontype     0
monthlychargesinr    0
totalchargesinr      0
contracttype         0
supporttickets       0
latepayments         0
paymentmethod        0
autopay              0
churn                0
dtype: int64

Final data types:
customerid            object
age                    int64
region                object
tenuremonths           int64
subscriptiontype      object
monthlychargesinr    float64
totalchargesinr      float64
contracttype          object
supporttickets         int64
latepayments           int64
paymentmethod         object
autopay               object
churn                 object
dtype: object


In [3]:

# Export the cleaned dataset
output_file = "/content/customer_churn_cleaned.csv"

clean_df.to_csv(output_file, index=False)

print("Cleaned CSV saved successfully!")
print("Final rows:", len(clean_df))
print("Final columns:", len(clean_df.columns))
print("Remaining missing values:", clean_df.isnull().sum().sum())


Cleaned CSV saved successfully!
Final rows: 600
Final columns: 13
Remaining missing values: 0


In [4]:

from google.colab import files

files.download("/content/customer_churn_cleaned.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>